# 02 – Trends over the run: when does the failure become visible?

Phase 1 exploration. Each snapshot is reduced to RMS and kurtosis per bearing, and the trends are
plotted over the whole run of all three tests.

Requires the trend tables from `python scripts/compute_trends.py data/raw` (about a minute). The
notebook is committed without outputs; the exported figures are in `docs/figures/`, the findings in
`docs/exploration.md`.

- **RMS** grows with the overall vibration energy, typically late, once the damage has spread.
- **Kurtosis** (3 for Gaussian noise) rises when a local defect produces short impacts, often
  earlier than RMS, and can fall back once the damage spreads and the signal becomes noise-like again.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from bearing_monitor import trends
from bearing_monitor.config import FIGURES_DIR, PROCESSED_DATA_DIR

TESTS = (1, 2, 3)
BEARINGS = (1, 2, 3, 4)
STATISTICS = ("rms", "kurtosis")

tables = {}
for test in TESTS:
    table = pd.read_parquet(PROCESSED_DATA_DIR / f"trends_test{test}.parquet")
    # Flat-lined files at the end (sensors already off) are not vibration data.
    tables[test] = table[~table["flatline"]].reset_index()
{test: len(table) for test, table in tables.items()}

## Settings

The **onset rule** is descriptive only: it shows where a trend leaves the band of its own early,
healthy part. It is computed per bearing and statistic with a band of mean + `K` standard
deviations over the reference window, a trailing median over `SMOOTH` snapshots and a minimum
duration of `SUSTAIN` snapshots (6 snapshots = 1 hour). The reference window uses only early
snapshots: `RUN_IN` snapshots are skipped, then `REFERENCE` snapshots are used, the same scheme
as the planned training window of phase 3.

`LITERATURE_ONSET` holds the degradation onset (file index) estimated by Zhang et al. (2019,
arXiv:1912.01096, table 2) with three methods, as (earliest, latest). For test 3 only one method
gave a result. It is used for comparison only.

In [ ]:
RUN_IN = 200
REFERENCE = 250
K, SMOOTH, SUSTAIN = 6.0, 6, 6
reference = slice(RUN_IN, RUN_IN + REFERENCE)

FAILED = {(1, 3): "inner race", (1, 4): "roller", (2, 1): "outer race", (3, 3): "outer race"}
LITERATURE_ONSET = {
    (1, 3): (1910, 2120),
    (1, 4): (1641, 1760),
    (2, 1): (539, 710),
    (3, 3): (2367, 2367),
}
COLORS = {1: "tab:blue", 2: "tab:orange", 3: "tab:green", 4: "tab:red"}

## Onset table

In [ ]:
rows = []
for test, table in tables.items():
    for bearing in BEARINGS:
        row = {
            "test": test,
            "bearing": bearing,
            "failed": FAILED.get((test, bearing), ""),
            "files": len(table),
        }
        for statistic in STATISTICS:
            values = trends.bearing_trend(table, bearing, statistic)
            for label, size in (("150", 150), ("250", 250), ("300", 300)):
                window = slice(RUN_IN, RUN_IN + size)
                row[f"{statistic}_{label}"] = trends.first_sustained_exceedance(
                    values, window, k=K, smooth=SMOOTH, sustain=SUSTAIN
                )
        row["literature"] = LITERATURE_ONSET.get((test, bearing))
        rows.append(row)
onsets = pd.DataFrame(rows).set_index(["test", "bearing"])
onsets

## Figure 1: all bearings, whole run

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 10), constrained_layout=True)
for row, (test, table) in enumerate(tables.items()):
    for col, statistic in enumerate(STATISTICS):
        ax = axes[row, col]
        ax.axvspan(0, RUN_IN, color="0.9", lw=0)
        ax.axvspan(reference.start, reference.stop, color="tab:green", alpha=0.08, lw=0)
        for bearing in BEARINGS:
            failed = (test, bearing) in FAILED
            label = f"bearing {bearing}" + (" (failed)" if failed else "")
            ax.plot(
                table["file_index"],
                trends.bearing_trend(table, bearing, statistic),
                lw=0.7 if failed else 0.5,
                color=COLORS[bearing],
                label=label,
            )
        if statistic == "kurtosis":
            ax.set_yscale("log")
        ax.set(
            title=f"Test {test}: {statistic.upper() if statistic == 'rms' else 'Kurtosis'}",
            xlabel="Snapshot index (10 min apart)",
            xlim=(0, len(table)),
        )
        ax.legend(fontsize=8, loc="upper left")
fig.suptitle("Grey: skipped run-in, green: healthy reference window", fontsize=10)
fig.savefig(FIGURES_DIR / "trends_overview.png", dpi=120)

## Figure 2: making the failure visible

The four failed bearings. The trend of each failed bearing is drawn over the other bearings of the
same test (grey) with the literature onset range (orange band), the onset found by the rule above
(dashed line) and the end of the test (= failure, black line).

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 7), constrained_layout=True)
for col, ((test, bearing), defect) in enumerate(FAILED.items()):
    table = tables[test]
    for row, statistic in enumerate(STATISTICS):
        ax = axes[row, col]
        for other in BEARINGS:
            if other != bearing:
                ax.plot(
                    table["file_index"],
                    trends.bearing_trend(table, other, statistic),
                    lw=0.4,
                    color="0.75",
                )
        values = trends.bearing_trend(table, bearing, statistic)
        ax.plot(table["file_index"], values, lw=0.7, color=COLORS[bearing])
        early, late = LITERATURE_ONSET[(test, bearing)]
        ax.axvspan(
            early,
            max(late, early + 8),
            color="tab:orange",
            alpha=0.3,
            lw=0,
            label="literature onset",
        )
        onset = onsets.loc[(test, bearing), f"{statistic}_250"]
        if pd.notna(onset):
            ax.axvline(onset, color="k", ls="--", lw=1, label=f"rule onset ({int(onset)})")
        ax.axvline(len(table) - 1, color="k", lw=1.5, label="failure (end of test)")
        if statistic == "kurtosis":
            ax.set_yscale("log")
        ax.set(
            xlim=(0, len(table)),
            xlabel="Snapshot index" if row else None,
            ylabel="RMS" if statistic == "rms" else "Kurtosis",
        )
        if row == 0:
            ax.set_title(f"Test {test}, bearing {bearing}: {defect}")
        ax.legend(fontsize=7, loc="upper left")
fig.savefig(FIGURES_DIR / "failure_visible.png", dpi=120)

## Zoom: the last part of each run

The same data on the last snapshots only, where the rise happens.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 3.5), constrained_layout=True)
for ax, (test, bearing) in zip(axes, FAILED, strict=True):
    table = tables[test]
    start = max(0, len(table) - 600)
    part = table.iloc[start:]
    ax.plot(part["file_index"], trends.bearing_trend(part, bearing, "rms"), color="tab:blue")
    ax2 = ax.twinx()
    ax2.plot(
        part["file_index"], trends.bearing_trend(part, bearing, "kurtosis"), color="tab:red", lw=0.6
    )
    ax.set(title=f"Test {test}, bearing {bearing}", xlabel="Snapshot index")
    ax.set_ylabel("RMS", color="tab:blue")
    ax2.set_ylabel("Kurtosis", color="tab:red")